# 06 — Topics and trends (Book 2, Chapter 6)
Embeddings (all-MiniLM-L6-v2) + BERTopic. Trends are shares of each year's output, not counts.

In [1]:
import numpy as np, pandas as pd, re
from pathlib import Path
ROOT = Path("..").resolve() if Path("../config.yml").exists() else Path(".").resolve()
RAW, FIN, LOGS, INT = ROOT/"data/raw", ROOT/"data/final", ROOT/"logs", ROOT/"data/interim"
pm = pd.read_json(sorted(RAW.glob("pubmed_*.jsonl"))[-1], lines=True, dtype={"pmid": str})
pm = pm[pm["abstract"].notna() & pm["year"].notna()].reset_index(drop=True)
docs = (pm["title"].fillna("") + ". " + pm["abstract"]).tolist()
print(len(docs), "records with an abstract")

11167 records with an abstract


In [2]:
from sentence_transformers import SentenceTransformer
emb_file = INT/"embeddings_minilm.npy"
if emb_file.exists(): emb = np.load(emb_file)
else:
    emb = SentenceTransformer("all-MiniLM-L6-v2").encode(docs, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    np.save(emb_file, emb)
emb.shape

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(11167, 384)

In [3]:
from bertopic import BERTopic
from umap import UMAP; from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
stop = list(CountVectorizer(stop_words="english").get_stop_words()) + ["malaria","vaccine","vaccines","plasmodium","study","studies","results","using","used"]
tm = BERTopic(umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=2026),
              hdbscan_model=HDBSCAN(min_cluster_size=60, metric="euclidean", prediction_data=True),
              vectorizer_model=CountVectorizer(stop_words=stop, ngram_range=(1,2), min_df=5))
topics, _ = tm.fit_transform(docs, emb)
info = tm.get_topic_info()
print("topics:", (info.Topic >= 0).sum(), "| outliers:", round((np.array(topics) == -1).mean(), 3))
info[["Topic","Count","Name"]].head(30)

topics: 30 | outliers: 0.472


,Topic,Count,Name
0,-1,5276,-1_falciparum_responses_infection_protein
1,0,565,0_transmissionblocking_transmission_mosquito_g...
2,1,343,1_peptides_epitopes_peptide_cell
3,2,321,2_travel_travelers_advice_travellers
4,3,286,3_invasion_erythrocyte_falciparum_erythrocytes
5,4,285,4_igg_antibody_antibodies_antigens
6,5,276,5_diversity_genetic_isolates_falciparum
7,6,265,6_var2csa_placental_women_pregnancy
8,7,265,7_parasitic_leishmaniasis_leishmania_schistoso...
9,8,249,8_antimalarial_drug_drugs_resistance


Outliers are reassigned to their nearest topic so that trends use every record; the share
left as outlier is reported above.

In [4]:
topics2 = tm.reduce_outliers(docs, topics, strategy="embeddings", embeddings=emb)
pm["topic"] = topics2
names = {r.Topic: r.Name for r in info.itertuples()}
pm["topic_name"] = pm["topic"].map(names)
pm["period"] = pd.cut(pm["year"], [0,1999,2009,2019,2026], labels=["≤1999","2000–09","2010–19","2020–26"])
share = pd.crosstab(pm["topic_name"], pm["period"], normalize="columns").round(3)
share["change"] = share["2020–26"] - share["2010–19"]
print(share.sort_values("change").head(6).to_string()); print(share.sort_values("change").tail(6).to_string())
pm[["pmid","year","topic","topic_name"]].to_parquet(FIN/"topics.parquet", index=False)
info.to_csv(LOGS/"topic_info.csv", index=False)

period                                                    ≤1999  2000–09  2010–19  2020–26  change
topic_name                                                                                        
12_merozoite surface_msp1_merozoite_surface               0.064    0.079    0.034    0.013  -0.021
17_adenovirus_vectors_responses_vector                    0.003    0.019    0.031    0.011  -0.020
2_travel_travelers_advice_travellers                      0.035    0.035    0.036    0.023  -0.013
1_peptides_epitopes_peptide_cell                          0.126    0.064    0.036    0.025  -0.011
0_transmissionblocking_transmission_mosquito_gametocytes  0.046    0.052    0.075    0.064  -0.011
4_igg_antibody_antibodies_antigens                        0.063    0.050    0.053    0.043  -0.010
period                                                ≤1999  2000–09  2010–19  2020–26  change
topic_name                                                                                    
21_mortality_healt

## Stability check: refit with a different seed and compare

In [5]:
tm2 = BERTopic(umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=7),
               hdbscan_model=HDBSCAN(min_cluster_size=60, prediction_data=True),
               vectorizer_model=CountVectorizer(stop_words=stop, ngram_range=(1,2), min_df=5))
t2, _ = tm2.fit_transform(docs, emb)
from sklearn.metrics import adjusted_rand_score
m = (np.array(topics) >= 0) & (np.array(t2) >= 0)
print("topics in refit:", len(set(t2)) - (1 if -1 in t2 else 0),
      "| adjusted Rand index (non-outliers in both):", round(adjusted_rand_score(np.array(topics)[m], np.array(t2)[m]), 2))

topics in refit: 31 | adjusted Rand index (non-outliers in both): 0.95


## A second lens: OpenAlex topics
OpenAlex assigns each work a primary topic, subfield and field with its own classifier.
Comparing it with our clusters shows where the two lenses agree, and what the odd clusters hold.

In [6]:
oe_file = FIN / "openalex_enrich.parquet"
if oe_file.exists():
    oe = pd.read_parquet(oe_file)[["pmid", "primary_topic", "primary_field"]]
    t = pm[["pmid", "topic", "topic_name", "year"]].merge(oe, on="pmid", how="left")
    print("records with an OpenAlex topic:", t.primary_topic.notna().mean().round(3))
    print("Top OpenAlex primary topics:\n", t.primary_topic.value_counts().head(10).to_string())
    print("\nOpenAlex fields:\n", t.primary_field.value_counts(normalize=True).head(6).round(3).to_string())
    for kw in ["travel", "leishman"]:
        lab = [n for n in t.topic_name.dropna().unique() if kw in n][0]
        sub = t[t.topic_name == lab]
        print(f"\nCluster '{lab}' ({len(sub)} papers): top OpenAlex topics\n", sub.primary_topic.value_counts().head(4).to_string())
    # Agreement: how concentrated is each BERTopic cluster in one OpenAlex topic?
    conc = t.dropna(subset=["primary_topic"]).groupby("topic_name").primary_topic.agg(lambda s: s.value_counts(normalize=True).iloc[0])
    print("\nMedian share of a cluster's papers in its most common OpenAlex topic:", round(conc.median(), 2))

records with an OpenAlex topic: 0.943
Top OpenAlex primary topics:
 primary_topic
Malaria Research and Control                 7309
Immunotherapy and Immune Responses            284
Travel-related health issues                  280
vaccines and immunoinformatics approaches     166
Vaccine Coverage and Hesitancy                162
Global Maternal and Child Health              128
Parasites and Host Interactions               112
T-cell and B-cell Immunology                   90
Immune responses and vaccinations              85
Mosquito-borne diseases and control            69

OpenAlex fields:
 primary_field
Medicine                                        0.801
Immunology and Microbiology                     0.080
Biochemistry, Genetics and Molecular Biology    0.048
Social Sciences                                 0.020
Agricultural and Biological Sciences            0.008
Pharmacology, Toxicology and Pharmaceutics      0.007

Cluster '2_travel_travelers_advice_travellers' (359 papers):